# Laboratorio 7. NLP end-to-end y Embeddings

CC3092 Deep Learning y Sistemas Inteligentes. Brandon Werner Rivera Cabrera 23088

Se construye el pipeline completo texto → tokens → IDs → vectores → modelo → salida. Sobre un subconjunto de WikiText-103 se entrena un skip-gram con negative sampling (SGNS) implementado en PyTorch, se valida contra el Word2Vec de gensim con el mismo corpus y vocabulario, y ambos se comparan con GloVe preentrenado (glove-wiki-gigaword-100). Después se verifica la aritmética vectorial con analogías y se usan los tres conjuntos de embeddings para clasificar noticias de AG News.

**Cómo se ejecuta.** El notebook corre de principio a fin sin intervención (`jupyter nbconvert --execute`). Los resultados quedan en `results/`, las figuras en `figs/` y los vectores del mejor modelo en `vectores/`. El corpus preprocesado se escribe en `datos/` (no se sube al repositorio).

## 0. Configuración

In [ ]:
import os, re, json, time, math, html, random, platform, collections, warnings, io, contextlib
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import spearmanr
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE
from datasets import load_dataset
import gensim
import gensim.downloader as api
from gensim.models import Word2Vec, KeyedVectors
from gensim.models.word2vec import LineSentence
from gensim.test.utils import datapath
import nltk
from nltk.tokenize import TreebankWordTokenizer

warnings.filterwarnings("ignore")
SMOKE = os.environ.get("LAB7_SMOKE") == "1"          # prueba rápida con pocos datos, solo para verificar que todo corre
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42

SUBSET_TOKENS = 400_000 if SMOKE else 25_000_000     # tokens del subconjunto de WikiText-103 (mínimo pedido: 20 M)
MIN_COUNT = 3 if SMOKE else 5                         # frecuencia mínima para entrar al vocabulario
EPOCHS = 1 if SMOKE else 5                            # épocas de cada iteración de SGNS
BATCH = 4096 if SMOKE else 16384                      # pares (centro, contexto) por paso
TOP_EVAL = 30_000                                     # vocabulario compartido para analogías
EPOCHS_CLF = 2 if SMOKE else 10                       # épocas del clasificador de AG News
CONTROL = ["king", "france", "computer", "good", "january", "run"]

def fijar_semilla(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
fijar_semilla()

def sincronizar():
    if DEVICE.type == "cuda": torch.cuda.synchronize()

for d in ["figs", "results", "vectores", "datos"]:
    os.makedirs(d, exist_ok=True)

entorno = {
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "sin GPU",
    "cpu": platform.processor() or platform.machine(),
    "nucleos_cpu": os.cpu_count(),
    "sistema": platform.platform(),
    "python": platform.python_version(),
    "torch": torch.__version__,
    "gensim": gensim.__version__,
    "smoke": SMOKE,
}
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False})
AZUL, NARANJA, VERDE, MORADO, GRIS = "#2a78d6", "#eb6834", "#1baf7a", "#8a5cd1", "#5f5e5a"
print(json.dumps(entorno, indent=2))

## 1. Datos

- **WikiText-103 (raw)**: artículos de Wikipedia en inglés sin reemplazo por `<unk>`. Se usa el split de entrenamiento como corpus.
- **AG News**: 120 000 noticias de entrenamiento y 7 600 de test en 4 categorías (World, Sports, Business, Sci/Tech).
- **GloVe** `glove-wiki-gigaword-100`: 400 000 palabras en minúsculas, 100 dimensiones, entrenado sobre 6 000 millones de tokens de Wikipedia y Gigaword.
- **Benchmarks** de gensim: 19 544 analogías en 14 categorías, WordSim-353 y SimLex-999.

In [ ]:
t0 = time.time()
corpus = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1")
news = load_dataset("fancyzhx/ag_news")
with contextlib.redirect_stdout(io.StringIO()):          # evita guardar la barra de progreso de la descarga
    glove = api.load("glove-wiki-gigaword-100")
print(corpus)
print(news)
print(f"GloVe: {len(glove.key_to_index):,} palabras, dimensión {glove.vector_size}")
print(f"carga: {time.time() - t0:.0f} s")

RUTA_ANALOGIAS = datapath("questions-words.txt")
RUTA_WS = datapath("wordsim353.tsv")
RUTA_SIMLEX = datapath("simlex999.txt")

## 2. Exploración y preprocesamiento del texto

### 2.1 Normalización y tokenizador

Decisiones, pensadas para que el vocabulario sea consistente con el de GloVe (que usa el tokenizador de Stanford sobre texto en minúsculas):

- **Marcadores de WikiText.** El texto raw separa guiones y separadores numéricos como `well @-@ known`, `1 @,@ 000` y `3 @.@ 5`. Se reconstruyen como `well-known`, `1,000` y `3.5`, que es como aparecen en GloVe.
- **Minúsculas.** GloVe 6B no distingue mayúsculas; si el corpus propio las distinguiera, `Paris` y `paris` serían palabras distintas y las preguntas de analogías no coincidirían.
- **Contracciones.** Se separan como en el Penn Treebank: `don't` → `do n't`, `it's` → `it 's`. GloVe tiene `n't` y `'s` como tokens.
- **Compuestos con guion y números.** `well-known`, `state-of-the-art`, `1950s`, `1,000` y `3.5` quedan como un solo token. Los números se conservan en dígitos porque GloVe los tiene así.
- **Abreviaturas.** `u.s.`, `e.g.` y abreviaturas comunes como `mr.`, `dr.`, `st.` o `jan.` quedan como un token con su punto, igual que en GloVe.
- **Puntuación.** Para entrenar embeddings se descarta: solo aporta contextos de baja información y ningún benchmark la evalúa. El tokenizador puede conservarla para compararlo con NLTK.
- **Tokens especiales.** No se agregan al corpus de embeddings: las palabras con frecuencia menor a `min_count` se eliminan antes de formar ventanas, como hace word2vec. En el clasificador las palabras fuera del vocabulario se descartan y los lotes se arman con offsets, por lo que no hace falta `<pad>`.
- **Encabezados.** Las líneas `= Título =` y `= = Sección = =` marcan artículos y secciones; se cuentan pero no se usan como texto. Cada párrafo es una oración para word2vec, así que las ventanas no cruzan párrafos.

In [ ]:
ARROBA = re.compile(r" @([-,.])@ ")
NT = re.compile(r"(\w)(n't)\b")
CLITICO = re.compile(r"(\w)('s|'re|'ve|'ll|'d|'m)\b")
_ABREV = r"(?:[^\W\d_]\.){2,}"                                 # u.s.  e.g.
_ABREV_COMUN = r"\b(?:mr|mrs|ms|dr|st|jr|sr|vs|prof|gen|col|lt|sgt|capt|gov|sen|rev|inc|ltd|corp|jan|feb|aug|sept|oct|nov|dec|mt|ft)\."
_CLIT = r"n't\b|'(?:s|re|ve|ll|d|m)\b"                        # clíticos ya separados
_PAL = r"[^\W_]+(?:(?:[-']|(?<=\d)[.,](?=\d))[^\W_]+)*"        # palabras, compuestos con guion y números 1,000.5
_PUNT = r"[^\w\s]"
RE_PALABRAS = re.compile("|".join([_ABREV_COMUN, _ABREV, _CLIT, _PAL]))
RE_CON_PUNT = re.compile("|".join([_ABREV_COMUN, _ABREV, _CLIT, _PAL, _PUNT]))

def normalizar(texto):
    t = ARROBA.sub(r"\1", texto).lower()
    t = NT.sub(r"\1 \2", t)
    t = CLITICO.sub(r"\1 \2", t)
    return CLITICO.sub(r"\1 \2", t)          # segunda pasada para casos como i'd've

def tokenizar(texto, puntuacion=False):
    return (RE_CON_PUNT if puntuacion else RE_PALABRAS).findall(normalizar(texto))

ENTIDAD = re.compile(r"#(\d+);")
def limpiar_ag(texto):
    # AG News trae entidades HTML sin '&' (#39; = apóstrofo) y barras invertidas
    t = ENTIDAD.sub(lambda m: chr(int(m.group(1))), texto)
    return html.unescape(t).replace("\\", " ")

print(tokenizar("The U.S. economy grew 3 @.@ 5 % in 2011 , and it's a well @-@ known fact that 1 @,@ 000 people don't agree ."))

### 2.2 Comparación con un tokenizador de librería

Se compara contra `TreebankWordTokenizer` de NLTK, que implementa las reglas del Penn Treebank y no necesita descargas. Ambos reciben el texto en minúsculas y conservan la puntuación para que la comparación sea token a token.

In [ ]:
ORACIONES_PRUEBA = [
    "I don't think it's ready, isn't it?",
    "The U.S. economy grew 3.5% in 2011.",
    "She's a well-known, state-of-the-art researcher.",
    "Dr. Smith paid $1,000.50 for the e-mail service.",
    "They'll arrive at 5 p.m. on Jan. 3rd, won't they?",
    "The company's CEO said: \"We can't wait.\"",
    "Rock 'n' roll was born in the 1950s.",
    "Visit https://www.example.com or write to info@example.com today.",
    "O'Neill scored in the 2-1 win against the U.K. team.",
    "I'd've gone if you'd asked, y'all know that.",
    "Mr. and Mrs. Brown live at 221B Baker St. in London.",
    "The temperature dropped to -5°C overnight (a record low).",
]
treebank = TreebankWordTokenizer()
filas = []
for s in ORACIONES_PRUEBA:
    propio = tokenizar(s, puntuacion=True)
    ref = treebank.tokenize(s.lower())
    filas.append({"oracion": s, "propio": " | ".join(propio), "nltk": " | ".join(ref),
                  "iguales": propio == ref,
                  "solo_propio": " ".join(sorted(set(propio) - set(ref))),
                  "solo_nltk": " ".join(sorted(set(ref) - set(propio)))})
tabla_tok = pd.DataFrame(filas)
tabla_tok.to_csv("results/tokenizadores.csv", index=False)
pd.set_option("display.max_colwidth", 120)
print(f"oraciones con tokenización idéntica: {tabla_tok.iguales.sum()} de {len(tabla_tok)}")
tabla_tok[["oracion", "solo_propio", "solo_nltk"]]

Los dos coinciden en contracciones (`do n't`, `it 's`, `ca n't`, `wo n't`), abreviaturas, números con separadores y compuestos con guion, que es lo que importa para el vocabulario de GloVe. Las diferencias quedan en las comillas, que Treebank convierte en ` `` ` y `''`, en `'n'` y en contracciones dobles como `i'd've`, en las URL y correos, que el tokenizador propio parte en sus palabras, y en números con signo y unidades como `-5°c`.

### 2.3 Tamaño del corpus

In [ ]:
textos = corpus["train"]["text"]
stats = collections.Counter()
cuenta_total = collections.Counter()
sub_oraciones, sub_tokens, en_subset = [], 0, True
t0 = time.time()
for linea in textos:
    stats["lineas"] += 1
    s = linea.strip()
    if not s:
        stats["lineas_vacias"] += 1
        continue
    stats["tokens_espacio_total"] += len(s.split())
    if s.startswith("= ") and s.endswith(" ="):
        if s.startswith("= = "):
            stats["secciones"] += 1
        else:
            stats["articulos"] += 1
            if sub_tokens >= SUBSET_TOKENS:
                en_subset = False          # el subconjunto termina en un límite de artículo
        continue
    stats["parrafos"] += 1
    toks = RE_PALABRAS.findall(normalizar(s))
    stats["tokens_palabra"] += len(toks)
    cuenta_total.update(toks)
    if en_subset:
        sub_oraciones.append(toks)
        sub_tokens += len(toks)
        stats["articulos_subset"] = stats["articulos"]
stats["vocabulario_total"] = len(cuenta_total)
stats["tokens_subset"] = sub_tokens
stats["parrafos_subset"] = len(sub_oraciones)
stats["lineas_validation"] = len(corpus["validation"])
stats["lineas_test"] = len(corpus["test"])
del textos
print(f"procesado en {time.time() - t0:.0f} s")
for k, v in stats.items():
    print(f"  {k:24s} {v:>14,}")
json.dump(dict(stats), open("results/corpus.json", "w"), indent=2)

### 2.4 Ley de Zipf y cobertura

In [ ]:
frec = np.array(sorted(cuenta_total.values(), reverse=True), dtype=np.float64)
rango = np.arange(1, len(frec) + 1)
total = frec.sum()
mask = (rango >= 10) & (rango <= min(100_000, len(frec)))
pendiente, intercepto = np.polyfit(np.log10(rango[mask]), np.log10(frec[mask]), 1)
cobertura = {k: float(frec[:k].sum() / total) for k in (10, 1000, 30000)}

fig, ax = plt.subplots(figsize=(6.5, 4))
sel = np.unique(np.logspace(0, np.log10(len(frec)), 2000).astype(int)) - 1
ax.loglog(rango[sel], frec[sel], ".", ms=3, color=AZUL, label="WikiText-103")
ax.loglog(rango[sel], 10 ** intercepto * rango[sel] ** pendiente, "--", color=NARANJA,
          label=f"ajuste: pendiente {pendiente:.2f}")
ax.set_xlabel("rango"); ax.set_ylabel("frecuencia")
ax.set_title("Frecuencia contra rango", loc="left", fontsize=10)
ax.legend(frameon=False)
fig.tight_layout(); fig.savefig("figs/zipf.png", bbox_inches="tight"); plt.show()

print(f"pendiente del ajuste log-log (rangos 10 a 100 000): {pendiente:.3f}")
for k, v in cobertura.items():
    print(f"las {k:>6,} palabras más frecuentes cubren {100 * v:.1f} % de los tokens")
print("10 más frecuentes:", [w for w, _ in cuenta_total.most_common(10)])
json.dump({"pendiente": pendiente, "cobertura": cobertura}, open("results/zipf.json", "w"), indent=2)

### 2.5 Vocabulario y umbral de frecuencia mínima

In [ ]:
cuenta_sub = collections.Counter(w for s in sub_oraciones for w in s)
filas = []
for mc in [1, 2, 3, 5, 10, 20, 50]:
    tam = sum(1 for c in cuenta_sub.values() if c >= mc)
    desconocidos = sum(c for c in cuenta_sub.values() if c < mc) / sub_tokens
    filas.append({"min_count": mc, "vocabulario": tam, "tokens_desconocidos_pct": 100 * desconocidos})
tabla_vocab = pd.DataFrame(filas)
tabla_vocab.to_csv("results/vocabulario_umbral.csv", index=False)

vocab = [w for w, c in cuenta_sub.most_common() if c >= MIN_COUNT]       # ordenado por frecuencia
w2i = {w: i for i, w in enumerate(vocab)}
V = len(vocab)
print(f"subconjunto: {sub_tokens:,} tokens en {len(sub_oraciones):,} párrafos")
print(f"min_count = {MIN_COUNT}: vocabulario de {V:,} palabras")
tabla_vocab

### 2.6 Corpus de entrenamiento como IDs

Cada párrafo se convierte en IDs y se eliminan las palabras fuera del vocabulario. Los párrafos de más de 1000 tokens se parten (gensim trunca oraciones de más de 10 000). El mismo corpus se escribe en `datos/corpus_sgns.txt` para que gensim entrene exactamente sobre los mismos tokens.

In [ ]:
ids_oraciones = []
for s in sub_oraciones:
    ids = np.fromiter((w2i[w] for w in s if w in w2i), dtype=np.int32)
    for i in range(0, len(ids), 1000):
        if len(ids[i:i + 1000]):
            ids_oraciones.append(ids[i:i + 1000])
del sub_oraciones
largos = np.array([len(x) for x in ids_oraciones])
TOKENS = np.concatenate(ids_oraciones)
ORACION = np.repeat(np.arange(len(ids_oraciones), dtype=np.int32), largos)
N_TOKENS = len(TOKENS)
CORTE_ACUM = np.cumsum(largos)

RUTA_CORPUS = "datos/corpus_sgns.txt"
with open(RUTA_CORPUS, "w", encoding="utf8") as f:
    for ids in ids_oraciones:
        f.write(" ".join(vocab[i] for i in ids) + "\n")
CUENTAS = np.bincount(TOKENS, minlength=V).astype(np.float64)
assert (CUENTAS >= MIN_COUNT).all()
print(f"tokens para entrenar: {N_TOKENS:,} en {len(ids_oraciones):,} oraciones "
      f"({100 * (1 - N_TOKENS / sub_tokens):.2f} % eliminados por min_count)")

### 2.7 Submuestreo de palabras frecuentes

word2vec descarta cada aparición de una palabra con probabilidad $1 - P(w)$, donde $P(w) = \left(\sqrt{f(w)/t} + 1\right) \cdot t / f(w)$, $f(w)$ es la frecuencia relativa y $t$ el umbral. Es la fórmula del código C original y de gensim. Las palabras muy frecuentes se descartan casi siempre, lo que reduce los pares poco informativos (`the`, `of`), acelera el entrenamiento y hace que las ventanas abarquen más palabras de contenido.

In [ ]:
def prob_conservar(cuentas, t):
    umbral = t * cuentas.sum()
    p = (np.sqrt(cuentas / umbral) + 1) * umbral / np.maximum(cuentas, 1)
    return np.minimum(p, 1.0)

filas = []
for t in [1e-3, 1e-4, 1e-5]:
    p = prob_conservar(CUENTAS, t)
    fila = {"t": t, "tokens_descartados_pct": 100 * float(((1 - p) * CUENTAS).sum() / CUENTAS.sum())}
    for w in ["the", "of", "and"]:
        fila[f"descarte_{w}_pct"] = 100 * float(1 - p[w2i[w]])
    filas.append(fila)
tabla_sub = pd.DataFrame(filas)
tabla_sub.to_csv("results/submuestreo.csv", index=False)
for w in ["the", "of", "and"]:
    print(f"{w:4s} aparece {int(CUENTAS[w2i[w]]):,} veces ({100 * CUENTAS[w2i[w]] / N_TOKENS:.2f} % del corpus)")
tabla_sub

### 2.8 Pares (centro, contexto) de skip-gram

Para cada token se toman como contexto las palabras a distancia menor o igual a la ventana dentro del mismo párrafo. word2vec usa una ventana dinámica: para cada centro se sortea un radio entre 1 y la ventana, lo que da más peso a las palabras cercanas. Los pares se generan directamente en la GPU en cada época.

In [ ]:
TOK_GPU = torch.from_numpy(TOKENS).to(DEVICE)
ORA_GPU = torch.from_numpy(ORACION).to(DEVICE)

def pares_epoch(tok, ora, p_conservar, ventana, generador=None):
    # submuestreo, ventana dinámica y pares dentro de la misma oración, todo en la GPU
    conservar = torch.rand(len(tok), device=tok.device, generator=generador) < p_conservar[tok.long()]
    t, o = tok[conservar], ora[conservar]
    radio = torch.randint(1, ventana + 1, (len(t),), device=tok.device, dtype=torch.int8, generator=generador)
    centros, contextos = [], []
    for d in range(1, ventana + 1):
        misma = o[:-d] == o[d:]
        adelante = misma & (radio[:-d] >= d)       # centro a la izquierda, contexto a la derecha
        atras = misma & (radio[d:] >= d)            # centro a la derecha, contexto a la izquierda
        centros += [t[:-d][adelante], t[d:][atras]]
        contextos += [t[d:][adelante], t[:-d][atras]]
    c, x = torch.cat(centros), torch.cat(contextos)
    perm = torch.randperm(len(c), device=tok.device, generator=generador)
    return c[perm], x[perm], int(conservar.sum())

def contar_pares_sin_submuestreo(ventana):
    fijos = dinamicos = 0
    for d in range(1, ventana + 1):
        misma = int((ORACION[:-d] == ORACION[d:]).sum())
        fijos += 2 * misma
        dinamicos += 2 * misma * (ventana - d + 1) / ventana
    return fijos, dinamicos

VENTANA_BASE, T_BASE = 5, 1e-4
fijos, dinamicos = contar_pares_sin_submuestreo(VENTANA_BASE)
p_base = torch.tensor(prob_conservar(CUENTAS, T_BASE), dtype=torch.float32, device=DEVICE)
c, x, conservados = pares_epoch(TOK_GPU, ORA_GPU, p_base, VENTANA_BASE)
pares = {"ventana": VENTANA_BASE, "t": T_BASE, "tokens": N_TOKENS, "tokens_tras_submuestreo": conservados,
         "pares_ventana_fija_sin_submuestreo": fijos, "pares_ventana_dinamica_sin_submuestreo": dinamicos,
         "pares_ventana_dinamica_con_submuestreo": len(c)}
del c, x
json.dump(pares, open("results/pares.json", "w"), indent=2)
for k, v in pares.items():
    print(f"  {k:42s} {v:>16,.0f}" if isinstance(v, (int, float)) and v > 1 else f"  {k:42s} {v}")

## 3. Investigación: embeddings en PyTorch y gensim

- **`nn.Embedding(num_embeddings, embedding_dim, padding_idx, sparse)`**: tabla de búsqueda de tamaño `num_embeddings × embedding_dim` cuyo forward devuelve las filas indicadas por los IDs. `padding_idx` fija una fila en ceros que no recibe gradiente. `sparse=True` produce gradientes dispersos con solo las filas usadas en el lote, lo que ahorra memoria y cómputo cuando el vocabulario es grande; requiere optimizadores como `SGD` o `SparseAdam`.
- **`nn.Embedding.from_pretrained(embeddings, freeze)`**: crea la tabla a partir de una matriz existente. Con `freeze=True` (el valor por defecto) la tabla no se entrena; con `False` se ajusta junto con el resto del modelo.
- **`nn.EmbeddingBag(num_embeddings, embedding_dim, mode)`**: busca y agrega en una sola operación los vectores de una bolsa de palabras sin materializar la matriz intermedia. `mode` puede ser `"sum"`, `"mean"` o `"max"`. Las secuencias de distinto largo se pasan concatenadas en un tensor 1D junto con `offsets`, la posición de inicio de cada documento, así que no hace falta padding.
- **CBOW y skip-gram**: CBOW predice la palabra central a partir del promedio de su contexto; es más rápido y suaviza las palabras frecuentes. Skip-gram predice cada palabra del contexto a partir de la central; genera más pares por token y representa mejor las palabras poco frecuentes.
- **Dos matrices en Word2Vec**: $W$ (entrada) representa una palabra cuando es el centro y $W'$ (salida) cuando es contexto o negativo. Separarlas evita que una palabra tenga que ser similar a sí misma, ya que casi nunca aparece en su propio contexto. Normalmente se conserva $W$; a veces se usa $W + W'$.
- **Word2Vec frente a GloVe**: Word2Vec es predictivo; recorre el corpus y ajusta los vectores para predecir contextos con SGD local. GloVe primero cuenta la matriz global de co-ocurrencias $X$ y luego ajusta $w_i^\top \tilde w_j + b_i + \tilde b_j \approx \log X_{ij}$ con mínimos cuadrados ponderados. Levy y Goldberg mostraron que SGNS factoriza implícitamente una matriz PMI desplazada, así que la diferencia práctica está más en los hiperparámetros que en el método.

In [ ]:
emb = nn.Embedding(10, 4, padding_idx=0)
print("fila de padding:", emb.weight[0].tolist())
pre = nn.Embedding.from_pretrained(torch.randn(5, 3), freeze=True)
print("from_pretrained con freeze=True, requires_grad =", pre.weight.requires_grad)

ids = torch.tensor([1, 2, 4, 5, 4, 3, 2, 9])
offsets = torch.tensor([0, 4])                    # dos documentos: ids[0:4] e ids[4:]
for modo in ["sum", "mean", "max"]:
    bolsa = nn.EmbeddingBag(10, 3, mode=modo)
    with torch.no_grad():
        bolsa.weight.copy_(torch.arange(30, dtype=torch.float32).view(10, 3))
    print(f"{modo:4s}", bolsa(ids, offsets).tolist())

## 4. Construcción y entrenamiento de los embeddings

### 4.1 Evaluación

Para que todas las evaluaciones usen exactamente las mismas preguntas, el vocabulario de analogías son las 30 000 palabras más frecuentes del corpus propio que también están en GloVe (el vocabulario de gensim es idéntico al propio). WordSim-353 y SimLex-999 se evalúan sobre los pares cuyas dos palabras están en ambos vocabularios. Las analogías usan 3CosAdd sobre vectores normalizados, excluyendo las tres palabras de la pregunta.

In [ ]:
PALABRAS_EVAL = [w for w in vocab if w in glove.key_to_index][:TOP_EVAL]
IDX_EVAL = {w: i for i, w in enumerate(PALABRAS_EVAL)}

CATEGORIAS, preguntas, cat_actual, total_preguntas = [], [], None, 0
for linea in open(RUTA_ANALOGIAS, encoding="utf8"):
    if linea.startswith(":"):
        cat_actual = linea[1:].strip(); CATEGORIAS.append(cat_actual); continue
    total_preguntas += 1
    a, b, c, d = linea.lower().split()
    if all(w in IDX_EVAL for w in (a, b, c, d)):
        preguntas.append((len(CATEGORIAS) - 1, IDX_EVAL[a], IDX_EVAL[b], IDX_EVAL[c], IDX_EVAL[d]))
Q = torch.tensor([p[1:] for p in preguntas], device=DEVICE)
Q_CAT = torch.tensor([p[0] for p in preguntas], device=DEVICE)
SEMANTICAS = torch.tensor([not c.startswith("gram") for c in CATEGORIAS], device=DEVICE)

def leer_pares(ruta):
    out = []
    for l in open(ruta, encoding="utf8"):
        if l.startswith("#") or not l.strip():
            continue
        a, b, s = l.lower().strip().split("\t")[:3]
        out.append((a, b, float(s)))
    return out
WS = [p for p in leer_pares(RUTA_WS) if p[0] in w2i and p[1] in w2i and p[0] in glove.key_to_index and p[1] in glove.key_to_index]
SIMLEX = [p for p in leer_pares(RUTA_SIMLEX) if p[0] in w2i and p[1] in w2i and p[0] in glove.key_to_index and p[1] in glove.key_to_index]
PALABRAS_PARES = sorted({w for a, b, _ in WS + SIMLEX for w in (a, b)})
IDX_PARES = {w: i for i, w in enumerate(PALABRAS_PARES)}
CONTROL_OK = [w for w in CONTROL if w in IDX_EVAL]

print(f"vocabulario de evaluación: {len(PALABRAS_EVAL):,} palabras")
print(f"analogías evaluables: {len(preguntas):,} de {total_preguntas:,} ({100 * len(preguntas) / total_preguntas:.1f} %)")
print(f"WordSim-353: {len(WS)} pares | SimLex-999: {len(SIMLEX)} pares")

def matriz(obtener, palabras):
    # filas alineadas a una lista de palabras; obtener(w) devuelve el vector de w
    return torch.tensor(np.stack([obtener(w) for w in palabras]), dtype=torch.float32, device=DEVICE)

FILAS_EVAL = torch.tensor([w2i[w] for w in PALABRAS_EVAL], device=DEVICE)
FILAS_PARES = torch.tensor([w2i[w] for w in PALABRAS_PARES], device=DEVICE)

class Embeddings:
    # vista uniforme de un conjunto de vectores para evaluarlo: vectores normalizados del vocabulario
    # de analogías (E) y de las palabras de WordSim y SimLex (P)
    def __init__(self, nombre, obtener=None, W=None):
        self.nombre = nombre
        if W is not None:      # matriz con filas en el orden del vocabulario propio
            W = torch.as_tensor(W, dtype=torch.float32, device=DEVICE)
            self.E, self.P = F.normalize(W[FILAS_EVAL], dim=1), F.normalize(W[FILAS_PARES], dim=1)
        else:
            self.E = F.normalize(matriz(obtener, PALABRAS_EVAL), dim=1)
            self.P = F.normalize(matriz(obtener, PALABRAS_PARES), dim=1)

@torch.no_grad()
def predecir_analogias(E, q, metodo="add", excluir=True, bloque=1024):
    preds = []
    for i in range(0, len(q), bloque):
        qq = q[i:i + bloque]
        a, b, c = E[qq[:, 0]], E[qq[:, 1]], E[qq[:, 2]]
        if metodo == "add":
            s = (b - a + c) @ E.T
        else:   # 3CosMul de Levy y Goldberg con cosenos llevados a [0, 1]
            sa, sb, sc = (a @ E.T + 1) / 2, (b @ E.T + 1) / 2, (c @ E.T + 1) / 2
            s = sb * sc / (sa + 1e-3)
        if excluir:
            s.scatter_(1, qq[:, :3], float("-inf"))
        preds.append(s.argmax(1))
    return torch.cat(preds)

@torch.no_grad()
def evaluar_analogias(emb, metodo="add"):
    ok = (predecir_analogias(emb.E, Q, metodo) == Q[:, 3]).float()
    por_cat = {}
    for k, cat in enumerate(CATEGORIAS):
        m = Q_CAT == k
        por_cat[cat] = (float(ok[m].mean()) if m.any() else float("nan"), int(m.sum()))
    sem = SEMANTICAS[Q_CAT]
    return {"semanticas": float(ok[sem].mean()), "sintacticas": float(ok[~sem].mean()),
            "total": float(ok.mean()), "por_categoria": por_cat}

@torch.no_grad()
def spearman(emb, pares):
    a = torch.tensor([IDX_PARES[p[0]] for p in pares], device=DEVICE)
    b = torch.tensor([IDX_PARES[p[1]] for p in pares], device=DEVICE)
    sims = (emb.P[a] * emb.P[b]).sum(1).cpu().numpy()
    return float(spearmanr(sims, [p[2] for p in pares]).correlation)

@torch.no_grad()
def vecinos(emb, palabras=None, k=5):
    out = {}
    for w in (palabras or CONTROL_OK):
        s = emb.E @ emb.E[IDX_EVAL[w]]
        s[IDX_EVAL[w]] = float("-inf")
        top = torch.topk(s, k)
        out[w] = [(PALABRAS_EVAL[i], round(float(v), 3)) for v, i in zip(top.values, top.indices)]
    return out

def evaluar(emb):
    an = evaluar_analogias(emb)
    return {"ana_sem": an["semanticas"], "ana_sin": an["sintacticas"], "ana_total": an["total"],
            "wordsim": spearman(emb, WS)}

### 4.2 Skip-gram con negative sampling en PyTorch

Dos tablas `nn.Embedding` dispersas: $W$ para la palabra central y $W'$ para contextos y negativos. Para cada par se maximiza $\log\sigma(u_o^\top v_c) + \sum_{k=1}^{K}\log\sigma(-u_k^\top v_c)$ con $K$ negativos muestreados de la distribución unigrama elevada a 0.75. La inicialización es la de word2vec: $W$ uniforme en $\pm 0.5/d$ y $W'$ en ceros. Se optimiza con `SparseAdam`, que solo actualiza las filas presentes en el lote, con un learning rate que decae linealmente hasta casi cero como en word2vec. Los negativos se muestrean con `searchsorted` sobre la distribución acumulada.

In [ ]:
class SGNS(nn.Module):
    def __init__(self, V, d):
        super().__init__()
        self.entrada = nn.Embedding(V, d, sparse=True)     # W
        self.salida = nn.Embedding(V, d, sparse=True)      # W'
        nn.init.uniform_(self.entrada.weight, -0.5 / d, 0.5 / d)
        nn.init.zeros_(self.salida.weight)

    def forward(self, centro, contexto, negativos):
        v = self.entrada(centro)                                   # (B, d)
        pos = (v * self.salida(contexto)).sum(-1)                  # (B,)
        neg = torch.bmm(self.salida(negativos), v.unsqueeze(2)).squeeze(2)   # (B, K)
        return -(F.logsigmoid(pos) + F.logsigmoid(-neg).sum(1)).mean()

def entrenar_sgns(nombre, cfg, verbose=True):
    fijar_semilla()
    gen = torch.Generator(device=DEVICE); gen.manual_seed(SEED)
    n = N_TOKENS if cfg["frac"] >= 1 else int(CORTE_ACUM[np.searchsorted(CORTE_ACUM, cfg["frac"] * N_TOKENS)])
    tok, ora = TOK_GPU[:n], ORA_GPU[:n]
    cuentas = np.bincount(TOKENS[:n], minlength=V).astype(np.float64)
    p_cons = torch.tensor(np.where(cuentas > 0, prob_conservar(cuentas, cfg["sample"]), 0), dtype=torch.float32, device=DEVICE)
    dist = cuentas ** 0.75
    cdf = torch.tensor(np.cumsum(dist / dist.sum()), dtype=torch.float32, device=DEVICE)

    modelo = SGNS(V, cfg["dim"]).to(DEVICE)
    opt = torch.optim.SparseAdam(modelo.parameters(), lr=cfg["lr"])
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

    c, x, _ = pares_epoch(tok, ora, p_cons, cfg["window"], gen)
    pasos_totales = cfg["epochs"] * math.ceil(len(c) / BATCH)
    paso, hist, curva, vec_hist, t_inicio = 0, [], [], {}, time.time()
    for ep in range(1, cfg["epochs"] + 1):
        sincronizar(); t0 = time.time()
        if ep > 1:
            c, x, _ = pares_epoch(tok, ora, p_cons, cfg["window"], gen)
        suma, nb, suma_tramo = torch.zeros((), device=DEVICE), 0, torch.zeros((), device=DEVICE)
        for i in range(0, len(c), BATCH):
            cc, xx = c[i:i + BATCH].long(), x[i:i + BATCH].long()
            neg = torch.searchsorted(cdf, torch.rand(len(cc), cfg["neg"], device=DEVICE, generator=gen)).clamp_(max=V - 1)
            for g in opt.param_groups:
                g["lr"] = cfg["lr"] * max(1e-4, 1 - paso / pasos_totales)
            perdida = modelo(cc, xx, neg)
            opt.zero_grad(set_to_none=True)
            perdida.backward()
            opt.step()
            suma += perdida.detach(); suma_tramo += perdida.detach(); nb += 1; paso += 1
            if paso % 500 == 0:
                curva.append({"iteracion": nombre, "paso": paso, "perdida": float(suma_tramo) / 500})
                suma_tramo.zero_()
        sincronizar(); t_ep = time.time() - t0
        emb = Embeddings(nombre, W=modelo.entrada.weight.detach())
        ev = evaluar(emb)
        vec_hist[ep] = vecinos(emb)
        hist.append({"iteracion": nombre, "epoch": ep, "perdida": float(suma) / nb, "pares": len(c),
                     **ev, "t_epoch": t_ep})
        if verbose:
            print(f"  [{nombre}] ep {ep}/{cfg['epochs']}  pares {len(c) / 1e6:6.1f} M  pérdida {float(suma) / nb:.4f}  "
                  f"analogías {ev['ana_total']:.3f} (sem {ev['ana_sem']:.3f}, sin {ev['ana_sin']:.3f})  "
                  f"WS {ev['wordsim']:.3f}  ({t_ep:.1f} s)", flush=True)
    del c, x
    pico = torch.cuda.max_memory_allocated() / 2**20 if DEVICE.type == "cuda" else float("nan")
    resumen = {"iteracion": nombre, **{k: cfg[k] for k in ["dim", "window", "neg", "sample", "lr", "epochs", "frac"]},
               "tokens": n, "pares_por_epoch": hist[-1]["pares"], "perdida_final": hist[-1]["perdida"],
               "ana_sem": hist[-1]["ana_sem"], "ana_sin": hist[-1]["ana_sin"], "ana_total": hist[-1]["ana_total"],
               "wordsim": hist[-1]["wordsim"], "puntaje": (hist[-1]["ana_total"] + hist[-1]["wordsim"]) / 2,
               "t_epoch_medio": float(np.mean([h["t_epoch"] for h in hist])), "t_total": time.time() - t_inicio,
               "memoria_pico_mb": pico}
    W = modelo.entrada.weight.detach().cpu().numpy()
    del modelo, opt
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    return resumen, hist, curva, vec_hist, W

### 4.3 Iteraciones

La configuración base usa dimensión 100, ventana 5, 5 negativos, umbral de submuestreo $10^{-4}$, learning rate 0.005 y 5 épocas sobre el subconjunto completo. Cada iteración cambia un hiperparámetro respecto a la base. Al final se arma la configuración final de dimensión 100 con el mejor valor de ventana, negativos, submuestreo y learning rate, elegidos con el promedio de la accuracy de analogías y la correlación en WordSim-353. SimLex-999 no se usa para seleccionar.

In [ ]:
BASE = dict(dim=100, window=5, neg=5, sample=1e-4, lr=0.005, epochs=EPOCHS, frac=1.0)
ITERACIONES = [
    ("SGNS-1", "configuración base", {}),
    ("SGNS-2", "dimensión 50", {"dim": 50}),
    ("SGNS-3", "dimensión 300", {"dim": 300}),
    ("SGNS-4", "25 % del corpus", {"frac": 0.25}),
    ("SGNS-5", "50 % del corpus", {"frac": 0.5}),
    ("SGNS-6", "ventana 2", {"window": 2}),
    ("SGNS-7", "ventana 10", {"window": 10}),
    ("SGNS-8", "15 negativos", {"neg": 15}),
    ("SGNS-9", "submuestreo 1e-3", {"sample": 1e-3}),
    ("SGNS-10", "submuestreo 1e-5", {"sample": 1e-5}),
    ("SGNS-11", "learning rate 0.002", {"lr": 0.002}),
    ("SGNS-12", "learning rate 0.01", {"lr": 0.01}),
]
CONFIGS, RES, HIST, CURVAS, VECINOS, VECTORES = {}, [], [], [], {}, {}
for nombre, desc, cambio in ITERACIONES:
    cfg = {**BASE, **cambio}
    CONFIGS[nombre] = cfg
    print(f"\n=== {nombre}: {desc}")
    r, h, cur, vh, W = entrenar_sgns(nombre, cfg)
    r["descripcion"] = desc
    RES.append(r); HIST += h; CURVAS += cur; VECINOS[nombre] = vh; VECTORES[nombre] = W
pd.DataFrame(RES).to_csv("results/iteraciones_sgns.csv", index=False)

In [ ]:
# configuración final: mejor valor de cada hiperparámetro explorado, dimensión 100 y corpus completo
res = {r["iteracion"]: r for r in RES}
grupos = {"window": ["SGNS-1", "SGNS-6", "SGNS-7"], "neg": ["SGNS-1", "SGNS-8"],
          "sample": ["SGNS-1", "SGNS-9", "SGNS-10"], "lr": ["SGNS-1", "SGNS-11", "SGNS-12"]}
FINAL = dict(BASE)
for hp, its in grupos.items():
    mejor = max(its, key=lambda n: res[n]["puntaje"])
    FINAL[hp] = CONFIGS[mejor][hp]
    print(f"{hp:7s}: " + ", ".join(f"{n} {CONFIGS[n][hp]} -> {res[n]['puntaje']:.3f}" for n in its) + f"  | elegido {FINAL[hp]}")
repetida = next((n for n, c in CONFIGS.items() if c == FINAL), None)
if repetida:
    print(f"\nla configuración final coincide con {repetida}; se reutiliza ese modelo")
    r = dict(res[repetida]); r["iteracion"] = "SGNS-final"; r["descripcion"] = f"mejores valores (igual a {repetida})"
    RES.append(r); VECTORES["SGNS-final"] = VECTORES[repetida]; VECINOS["SGNS-final"] = VECINOS[repetida]
    HIST += [{**h, "iteracion": "SGNS-final"} for h in HIST if h["iteracion"] == repetida]
    CURVAS += [{**c, "iteracion": "SGNS-final"} for c in CURVAS if c["iteracion"] == repetida]
else:
    print(f"\n=== SGNS-final: {FINAL}")
    r, h, cur, vh, W = entrenar_sgns("SGNS-final", FINAL)
    r["descripcion"] = "mejores valores de cada hiperparámetro"
    RES.append(r); HIST += h; CURVAS += cur; VECINOS["SGNS-final"] = vh; VECTORES["SGNS-final"] = W
CONFIGS["SGNS-final"] = FINAL

tabla_iter = pd.DataFrame(RES)
tabla_iter.to_csv("results/iteraciones_sgns.csv", index=False)
pd.DataFrame(HIST).to_csv("results/historial_sgns.csv", index=False)
pd.DataFrame(CURVAS).to_csv("results/curva_perdida_sgns.csv", index=False)
json.dump(VECINOS, open("results/vecinos_por_epoch.json", "w"), indent=1, ensure_ascii=False)
tabla_iter[["iteracion", "descripcion", "dim", "window", "neg", "sample", "lr", "tokens", "pares_por_epoch",
            "perdida_final", "ana_sem", "ana_sin", "ana_total", "wordsim", "t_epoch_medio", "t_total", "memoria_pico_mb"]].round(4)

### 4.4 Curvas de pérdida y de analogías

In [ ]:
hist = pd.DataFrame(HIST)
cur = pd.DataFrame(CURVAS)
nombres = [n for n, _, _ in ITERACIONES] + ["SGNS-final"]
cmap = plt.get_cmap("tab20")
fig, axs = plt.subplots(1, 3, figsize=(16, 4.5))
for k, n in enumerate(nombres):
    h = hist[hist.iteracion == n]
    color = "black" if n == "SGNS-final" else cmap(k)
    axs[0].plot(h.epoch, h.perdida, marker="o", ms=3, color=color, label=n)
    axs[1].plot(h.epoch, h.ana_total, marker="o", ms=3, color=color, label=n)
    cc = cur[cur.iteracion == n]
    if len(cc):
        axs[2].plot(cc.paso, cc.perdida, lw=1, color=color, label=n)
axs[0].set_title("Pérdida promedio por época", loc="left", fontsize=10); axs[0].set_xlabel("época")
axs[1].set_title("Accuracy de analogías por época", loc="left", fontsize=10); axs[1].set_xlabel("época")
axs[2].set_title("Pérdida cada 500 pasos", loc="left", fontsize=10); axs[2].set_xlabel("paso")
axs[1].legend(fontsize=7, ncol=2, frameon=False)
fig.tight_layout(); fig.savefig("figs/curvas_sgns.png", bbox_inches="tight"); plt.show()

print("Vecinos de las palabras de control al final de cada época (SGNS-final):")
for ep, vs in VECINOS["SGNS-final"].items():
    print(f"  época {ep}: " + " | ".join(f"{w}: {', '.join(p for p, _ in v)}" for w, v in vs.items()))

### 4.5 Word2Vec de gensim con la configuración final

Mismo corpus (el archivo `datos/corpus_sgns.txt`), mismo `min_count` y por lo tanto el mismo vocabulario, y los hiperparámetros equivalentes a la configuración final. gensim entrena en CPU con SGD por par (Hogwild) y learning rate que decae de 0.025 a 0.0001.

In [ ]:
t0 = time.time()
w2v = Word2Vec(sentences=LineSentence(RUTA_CORPUS), vector_size=FINAL["dim"], window=FINAL["window"],
               negative=FINAL["neg"], sample=FINAL["sample"], min_count=MIN_COUNT, sg=1, epochs=FINAL["epochs"],
               workers=os.cpu_count(), seed=SEED, alpha=0.025, min_alpha=0.0001)
T_GENSIM = time.time() - t0
assert set(w2v.wv.index_to_key) == set(vocab), "el vocabulario de gensim debe ser idéntico"
print(f"gensim: {T_GENSIM:.0f} s con {os.cpu_count()} hilos | vocabulario idéntico: {len(w2v.wv.index_to_key):,} palabras")

EMB = {
    "SGNS": Embeddings("SGNS", W=VECTORES["SGNS-final"]),
    "gensim": Embeddings("gensim", lambda w: w2v.wv[w]),
    "GloVe": Embeddings("GloVe", lambda w: glove[w]),
}
eval_modelos = {}
for n, e in EMB.items():
    ev = evaluar(e)
    eval_modelos[n] = ev
    print(f"{n:7s} analogías {ev['ana_total']:.3f} (sem {ev['ana_sem']:.3f}, sin {ev['ana_sin']:.3f})  WS {ev['wordsim']:.3f}")

# vectores del mejor modelo en formato KeyedVectors de gensim
kv_sgns = KeyedVectors(vector_size=FINAL["dim"])
kv_sgns.add_vectors(vocab, VECTORES["SGNS-final"])
kv_sgns.save("vectores/sgns_final.kv")
print("tamaño de los vectores guardados:", sum(os.path.getsize(os.path.join("vectores", f)) for f in os.listdir("vectores")) // 2**20, "MB")

## 5. Verificación de la aritmética vectorial

### 5.1 Analogías individuales

`analogia(a, b, c, k)` resuelve «a es a b como c es a ?» con 3CosAdd: el vector $\hat b - \hat a + \hat c$ con vectores normalizados, similitud coseno contra todo el vocabulario compartido y exclusión de a, b y c. Se compara contra `most_similar(positive=[b, c], negative=[a])` de gensim sobre el mismo vocabulario.

In [ ]:
def analogia(a, b, c, k=5, emb=None, excluir=True):
    E = emb.E
    v = F.normalize(E[IDX_EVAL[b]] - E[IDX_EVAL[a]] + E[IDX_EVAL[c]], dim=0)
    s = E @ v
    if excluir:
        for w in (a, b, c):
            s[IDX_EVAL[w]] = float("-inf")
    top = torch.topk(s, k)
    return [(PALABRAS_EVAL[i], float(val)) for val, i in zip(top.values, top.indices)], s, v

ANALOGIAS_PROPIAS = [
    ("género", "man", "king", "woman", "queen"), ("género", "man", "father", "woman", "mother"),
    ("género", "boy", "brother", "girl", "sister"),
    ("país-capital", "france", "paris", "italy", "rome"), ("país-capital", "japan", "tokyo", "germany", "berlin"),
    ("país-capital", "spain", "madrid", "russia", "moscow"),
    ("país-gentilicio", "france", "french", "germany", "german"), ("país-gentilicio", "china", "chinese", "japan", "japanese"),
    ("país-gentilicio", "mexico", "mexican", "canada", "canadian"),
    ("comparativo", "good", "better", "bad", "worse"), ("comparativo", "big", "bigger", "small", "smaller"),
    ("superlativo", "fast", "fastest", "strong", "strongest"),
    ("tiempo verbal", "walk", "walked", "run", "ran"), ("tiempo verbal", "go", "went", "see", "saw"),
    ("tiempo verbal", "play", "playing", "sing", "singing"),
    ("plural", "car", "cars", "child", "children"), ("plural", "city", "cities", "mouse", "mice"),
]
ANALOGIAS_PROPIAS = [x for x in ANALOGIAS_PROPIAS if all(w in IDX_EVAL for w in x[1:])]

def kv_eval(emb):
    kv = KeyedVectors(vector_size=emb.E.shape[1])
    kv.add_vectors(PALABRAS_EVAL, emb.E.cpu().numpy())
    return kv
KV = {n: kv_eval(e) for n, e in EMB.items()}

filas, coincide, coincide_top1, sin_excluir = [], 0, 0, []
for n, emb in EMB.items():
    for tipo, a, b, c, d in ANALOGIAS_PROPIAS:
        top, s, v = analogia(a, b, c, 5, emb)
        ref = [w for w, _ in KV[n].most_similar(positive=[b, c], negative=[a], topn=5)]
        coincide += [w for w, _ in top] == ref
        coincide_top1 += top[0][0] == ref[0]
        rango = int((s > s[IDX_EVAL[d]]).sum()) + 1
        filas.append({"modelo": n, "tipo": tipo, "analogia": f"{a}:{b} :: {c}:?", "esperada": d,
                      "top5": ", ".join(f"{w} {sim:.2f}" for w, sim in top), "acierta": top[0][0] == d,
                      "rango_correcta": rango, "cos_correcta": float(emb.E[IDX_EVAL[d]] @ v)})
        top_ne, _, _ = analogia(a, b, c, 1, emb, excluir=False)
        sin_excluir.append({"modelo": n, "analogia": f"{a}:{b} :: {c}:?", "primera_sin_excluir": top_ne[0][0],
                            "es_consulta": top_ne[0][0] in (a, b, c), "es_b": top_ne[0][0] == b, "es_c": top_ne[0][0] == c})
tabla_an = pd.DataFrame(filas)
tabla_ne = pd.DataFrame(sin_excluir)
tabla_an.to_csv("results/analogias_individuales.csv", index=False)
tabla_ne.to_csv("results/analogias_sin_exclusion.csv", index=False)
print(f"respuesta idéntica a most_similar de gensim en {coincide_top1} de {len(filas)} consultas; top-5 idéntico en {coincide}")
json.dump({"consultas": len(filas), "top1_igual": coincide_top1, "top5_igual": coincide}, open("results/verificacion_most_similar.json", "w"))
print(tabla_an.groupby("modelo").acierta.agg(["sum", "count"]))
print("\nsin excluir a, b y c, la primera palabra es una de la consulta en:")
print(tabla_ne.groupby("modelo")[["es_consulta", "es_b", "es_c"]].mean().round(3))
tabla_an[tabla_an.modelo == "SGNS"][["tipo", "analogia", "esperada", "top5", "rango_correcta", "cos_correcta"]]

### 5.2 Paralelismo de los vectores diferencia

Si la aritmética fuera exacta, todos los pares de una misma relación tendrían el mismo vector diferencia. Para cada relación se calcula la similitud coseno promedio entre los vectores diferencia $\hat b - \hat a$ de todos sus pares, y se compara con pares de palabras al azar.

In [ ]:
RELACIONES = {
    "género": [("man", "woman"), ("king", "queen"), ("boy", "girl"), ("father", "mother"), ("brother", "sister"),
               ("son", "daughter"), ("husband", "wife"), ("uncle", "aunt"), ("he", "she"), ("prince", "princess")],
    "país-capital": [("france", "paris"), ("germany", "berlin"), ("italy", "rome"), ("japan", "tokyo"), ("china", "beijing"),
                     ("russia", "moscow"), ("spain", "madrid"), ("egypt", "cairo"), ("greece", "athens"), ("canada", "ottawa")],
    "plural": [("car", "cars"), ("dog", "dogs"), ("city", "cities"), ("child", "children"), ("house", "houses"),
               ("book", "books"), ("tree", "trees"), ("man", "men"), ("woman", "women"), ("year", "years")],
    "pasado": [("walk", "walked"), ("go", "went"), ("see", "saw"), ("take", "took"), ("play", "played"),
               ("write", "wrote"), ("run", "ran"), ("give", "gave"), ("make", "made"), ("say", "said")],
    "comparativo": [("good", "better"), ("bad", "worse"), ("big", "bigger"), ("small", "smaller"), ("fast", "faster"),
                    ("strong", "stronger"), ("old", "older"), ("young", "younger"), ("high", "higher"), ("long", "longer")],
}
rng = np.random.default_rng(SEED)
azar = [tuple(rng.choice(PALABRAS_EVAL[:5000], 2, replace=False)) for _ in range(10)]

@torch.no_grad()
def paralelismo(emb, pares):
    pares = [(a, b) for a, b in pares if a in IDX_EVAL and b in IDX_EVAL]
    D = F.normalize(torch.stack([emb.E[IDX_EVAL[b]] - emb.E[IDX_EVAL[a]] for a, b in pares]), dim=1)
    S = D @ D.T
    n = len(pares)
    return float((S.sum() - n) / (n * (n - 1))), n

filas = []
for n, emb in EMB.items():
    for rel, pares in list(RELACIONES.items()) + [("pares al azar", azar)]:
        cos, k = paralelismo(emb, pares)
        filas.append({"modelo": n, "relacion": rel, "pares": k, "cos_promedio": cos})
tabla_par = pd.DataFrame(filas)
tabla_par.to_csv("results/paralelismo.csv", index=False)
tabla_par.pivot(index="relacion", columns="modelo", values="cos_promedio").round(3)

### 5.3 Evaluación sistemática por categoría

Los tres modelos se evalúan sobre las mismas preguntas de `questions-words.txt` (todas sus palabras dentro de las 30 000 compartidas), con 3CosAdd y 3CosMul.

In [ ]:
filas = []
for n, emb in EMB.items():
    for metodo in ["add", "mul"]:
        r = evaluar_analogias(emb, metodo)
        for cat, (acc, cnt) in r["por_categoria"].items():
            filas.append({"modelo": n, "metodo": "3CosAdd" if metodo == "add" else "3CosMul", "categoria": cat,
                          "preguntas": cnt, "accuracy": acc})
        for k in ["semanticas", "sintacticas", "total"]:
            filas.append({"modelo": n, "metodo": "3CosAdd" if metodo == "add" else "3CosMul", "categoria": k.upper(),
                          "preguntas": None, "accuracy": r[k]})
tabla_cat = pd.DataFrame(filas)
tabla_cat.to_csv("results/analogias_categoria.csv", index=False)
COBERTURA = {"evaluadas": len(preguntas), "total": total_preguntas, "cobertura": len(preguntas) / total_preguntas}
json.dump(COBERTURA, open("results/cobertura_analogias.json", "w"), indent=2)
print(f"cobertura: {len(preguntas):,} / {total_preguntas:,} = {100 * COBERTURA['cobertura']:.1f} %")
tabla_cat.pivot_table(index="categoria", columns=["metodo", "modelo"], values="accuracy", sort=False).round(3)

### 5.4 Visualización con t-SNE

In [ ]:
GRUPOS = {
    "países": "france germany italy spain portugal russia china japan india brazil mexico canada australia egypt greece turkey iran iraq israel poland sweden norway denmark finland ireland scotland england netherlands belgium austria switzerland hungary romania ukraine argentina chile peru colombia venezuela cuba korea vietnam thailand indonesia philippines pakistan afghanistan nigeria kenya ethiopia morocco algeria libya sudan syria lebanon jordan iceland serbia croatia bulgaria",
    "ciudades": "london paris berlin rome madrid moscow tokyo beijing cairo athens vienna dublin chicago boston toronto sydney amsterdam brussels lisbon prague warsaw budapest istanbul delhi mumbai shanghai seoul melbourne montreal glasgow manchester liverpool",
    "números": "one two three four five six seven eight nine ten eleven twelve thirteen fourteen fifteen sixteen seventeen eighteen nineteen twenty thirty forty fifty sixty seventy eighty ninety hundred thousand million billion",
    "meses y días": "january february march april june july august september october november december monday tuesday wednesday thursday friday saturday sunday",
    "animales": "dog cat horse cow pig sheep goat chicken duck goose eagle hawk owl parrot fish shark whale dolphin seal lion tiger leopard wolf fox bear deer elk moose rabbit mouse rat squirrel monkey ape gorilla elephant giraffe zebra camel snake lizard frog turtle crocodile bee ant spider butterfly",
    "verbos": "walk jump swim fly drive ride climb throw catch eat drink sleep write read speak talk sing dance build destroy create buy sell give take bring send receive open close begin finish win lose fight kill attack defend teach learn study think believe know understand remember forget love hate help",
    "profesiones": "doctor nurse teacher lawyer engineer scientist soldier farmer writer poet painter musician singer actor actress architect pilot sailor priest bishop judge politician journalist photographer chef baker carpenter",
    "colores": "red blue green yellow orange purple pink brown black white gray grey violet crimson scarlet",
    "deportes": "football baseball basketball hockey tennis golf cricket rugby boxing wrestling swimming cycling volleyball soccer athletics skiing",
    "familia": "mother father son daughter brother sister uncle aunt cousin grandmother grandfather wife husband nephew niece",
    "comida": "bread rice meat cheese milk butter sugar salt apple banana potato tomato wine beer coffee tea soup cake",
    "música": "guitar piano violin drum trumpet flute saxophone cello organ harp clarinet",
}
etiq, palabras_tsne = [], []
for g, ws in GRUPOS.items():
    for w in ws.split():
        if w in IDX_EVAL and w not in palabras_tsne:
            palabras_tsne.append(w); etiq.append(g)
X = EMB["SGNS"].E[[IDX_EVAL[w] for w in palabras_tsne]].cpu().numpy()
Y = TSNE(n_components=2, perplexity=min(30, len(X) - 1), init="pca", metric="cosine", random_state=SEED).fit_transform(X)
fig, ax = plt.subplots(figsize=(11, 8))
cmap = plt.get_cmap("tab20")
for k, g in enumerate(GRUPOS):
    m = np.array(etiq) == g
    ax.scatter(Y[m, 0], Y[m, 1], s=14, color=cmap(k), label=g)
for i in range(0, len(palabras_tsne), 3):
    ax.annotate(palabras_tsne[i], Y[i], fontsize=6, alpha=0.7)
ax.legend(fontsize=8, frameon=False, markerscale=1.5, loc="best")
ax.set_title(f"t-SNE de {len(palabras_tsne)} palabras (SGNS final)", loc="left", fontsize=10)
ax.set_xticks([]); ax.set_yticks([])
fig.tight_layout(); fig.savefig("figs/tsne.png", bbox_inches="tight"); plt.show()

## 6. Pipeline end-to-end: clasificación de AG News

Mismo preprocesamiento y tokenizador de la sección 2, después de reconstruir las entidades HTML de AG News. Se separa el 10 % del entrenamiento como validación estratificada y el test oficial se usa solo al final.

- **Vocabulario del clasificador**: palabras que aparecen al menos 2 veces en el entrenamiento. Con embeddings preentrenados se usan solo las que existen en ese conjunto; las demás se descartan del documento.
- **Modelo**: `nn.EmbeddingBag` con promedio, seguido de un MLP de una capa oculta de 256 unidades con dropout 0.3. Los lotes se arman con offsets, sin padding.

In [ ]:
ETIQUETAS = ["World", "Sports", "Business", "Sci/Tech"]
t0 = time.time()
doc_tr = [tokenizar(limpiar_ag(t)) for t in news["train"]["text"]]
doc_te = [tokenizar(limpiar_ag(t)) for t in news["test"]["text"]]
y_todo = np.array(news["train"]["label"]); y_te = np.array(news["test"]["label"])
idx_tr, idx_va = train_test_split(np.arange(len(doc_tr)), test_size=0.1, stratify=y_todo, random_state=SEED)
print(f"tokenizado en {time.time() - t0:.0f} s | train {len(idx_tr):,}  val {len(idx_va):,}  test {len(doc_te):,}")

tokens_ag = [w for i in idx_tr for w in doc_tr[i]]
CONJUNTOS = {"SGNS": set(vocab), "gensim": set(w2v.wv.key_to_index), "GloVe": set(glove.key_to_index)}
OOV = {n: 100 * sum(w not in s for w in tokens_ag) / len(tokens_ag) for n, s in CONJUNTOS.items()}
for n, v in OOV.items():
    print(f"tokens de AG News fuera del vocabulario de {n}: {v:.2f} %")
cuenta_ag = collections.Counter(tokens_ag)
VOCAB_AG = [w for w, c in cuenta_ag.most_common() if c >= 2]
print(f"vocabulario de AG News con frecuencia >= 2: {len(VOCAB_AG):,} palabras")

In [ ]:
def codificar(palabras):
    idx = {w: i for i, w in enumerate(palabras)}
    def cod(docs):
        ids = [[idx[w] for w in d if w in idx] for d in docs]
        lens = np.array([len(x) for x in ids])
        flat = np.fromiter((i for x in ids for i in x), dtype=np.int64, count=int(lens.sum()))
        starts = np.concatenate([[0], np.cumsum(lens)[:-1]])
        return {k: torch.tensor(v, device=DEVICE) for k, v in [("flat", flat), ("lens", lens), ("starts", starts)]}
    return {"tr": cod([doc_tr[i] for i in idx_tr]), "va": cod([doc_tr[i] for i in idx_va]), "te": cod(doc_te),
            "V": len(palabras)}

FUENTES = {"SGNS": lambda w: VECTORES["SGNS-final"][w2i[w]], "gensim": lambda w: w2v.wv[w], "GloVe": lambda w: glove[w]}
VOCABS = {"Aleatorio": VOCAB_AG}
for n in FUENTES:
    VOCABS[n] = [w for w in VOCAB_AG if w in CONJUNTOS[n]]
COD = {n: codificar(p) for n, p in VOCABS.items()}
PESOS = {n: torch.tensor(np.stack([FUENTES[n](w) for w in VOCABS[n]]), dtype=torch.float32) for n in FUENTES}
Y = {"tr": torch.tensor(y_todo[idx_tr], device=DEVICE), "va": torch.tensor(y_todo[idx_va], device=DEVICE),
     "te": torch.tensor(y_te, device=DEVICE)}
for n, v in VOCABS.items():
    print(f"{n:10s} vocabulario del clasificador: {len(v):,}")

def lote(cod, docs):
    lens, starts = cod["lens"][docs], cod["starts"][docs]
    offs = torch.cumsum(lens, 0) - lens
    total = int(lens.sum())
    pos = torch.repeat_interleave(starts - offs, lens, output_size=total) + torch.arange(total, device=DEVICE)
    return cod["flat"][pos], offs

class Clasificador(nn.Module):
    def __init__(self, V, d=100, pesos=None, congelar=False, oculta=256, dropout=0.3):
        super().__init__()
        self.emb = nn.EmbeddingBag(V, d, mode="mean")
        if pesos is not None:
            with torch.no_grad():
                self.emb.weight.copy_(pesos)
        self.emb.weight.requires_grad_(not congelar)
        self.mlp = nn.Sequential(nn.Linear(d, oculta), nn.ReLU(), nn.Dropout(dropout), nn.Linear(oculta, 4))
    def forward(self, ids, offs):
        return self.mlp(self.emb(ids, offs))

@torch.no_grad()
def predecir(modelo, cod, bloque=4096):
    modelo.eval()
    n = len(cod["lens"]); salidas = []
    for i in range(0, n, bloque):
        ids, offs = lote(cod, torch.arange(i, min(n, i + bloque), device=DEVICE))
        salidas.append(modelo(ids, offs))
    return torch.cat(salidas)

def metricas(y, logits):
    pred = logits.argmax(1).cpu().numpy(); y = y.cpu().numpy()
    p, r, f, _ = precision_recall_fscore_support(y, pred, average="macro", zero_division=0)
    return {"acc": accuracy_score(y, pred), "precision": p, "recall": r, "f1": f}

def entrenar_clf(nombre, fuente, congelar, docs_tr=None, epochs=EPOCHS_CLF, lr=1e-3, batch=256, verbose=True):
    fijar_semilla()
    cod = COD[fuente]
    modelo = Clasificador(cod["V"], 100, PESOS.get(fuente), congelar).to(DEVICE)
    opt = torch.optim.Adam([p for p in modelo.parameters() if p.requires_grad], lr=lr)
    docs_tr = torch.arange(len(cod["tr"]["lens"]), device=DEVICE) if docs_tr is None else docs_tr
    ytr = Y["tr"]
    mejor, estado, hist, t0 = -1, None, [], time.time()
    for ep in range(1, epochs + 1):
        modelo.train()
        perm = docs_tr[torch.randperm(len(docs_tr), device=DEVICE)]
        suma, nb = torch.zeros((), device=DEVICE), 0
        for i in range(0, len(perm), batch):
            d = perm[i:i + batch]
            ids, offs = lote(cod["tr"], d)
            perdida = F.cross_entropy(modelo(ids, offs), ytr[d])
            opt.zero_grad(set_to_none=True); perdida.backward(); opt.step()
            suma += perdida.detach(); nb += 1
        logits = predecir(modelo, cod["va"])
        m = metricas(Y["va"], logits)
        hist.append({"modelo": nombre, "epoch": ep, "perdida_train": float(suma) / nb,
                     "perdida_val": float(F.cross_entropy(logits, Y["va"])), **m})
        if m["f1"] > mejor:
            mejor, estado, mejor_ep = m["f1"], {k: v.detach().clone() for k, v in modelo.state_dict().items()}, ep
        if verbose:
            print(f"  [{nombre}] ep {ep:2d}  train {hist[-1]['perdida_train']:.4f}  val {hist[-1]['perdida_val']:.4f}  "
                  f"acc {m['acc']:.4f}  f1 {m['f1']:.4f}", flush=True)
    modelo.load_state_dict(estado)
    total = sum(p.numel() for p in modelo.parameters())
    entren = sum(p.numel() for p in modelo.parameters() if p.requires_grad)
    resumen = {"modelo": nombre, "fuente": fuente, "congelado": congelar, "mejor_epoch": mejor_ep,
               **{f"val_{k}": v for k, v in hist[mejor_ep - 1].items() if k in ("acc", "precision", "recall", "f1")},
               "params_totales": total, "params_entrenables": entren, "t_total": time.time() - t0}
    return resumen, hist, modelo

### 6.1 Baseline TF-IDF y modelos con embeddings

In [ ]:
def analizador(toks):
    return toks + [a + " " + b for a, b in zip(toks, toks[1:])]      # unigramas y bigramas

t0 = time.time()
tfidf = TfidfVectorizer(analyzer=analizador, min_df=2, sublinear_tf=True, max_features=300_000)
X_tr = tfidf.fit_transform([doc_tr[i] for i in idx_tr])
X_va = tfidf.transform([doc_tr[i] for i in idx_va]); X_te = tfidf.transform(doc_te)
mejor_C, mejor_f1 = None, -1
for Cc in [1.0, 10.0]:
    lr_ = LogisticRegression(C=Cc, max_iter=2000).fit(X_tr, y_todo[idx_tr])
    p, r, f, _ = precision_recall_fscore_support(y_todo[idx_va], lr_.predict(X_va), average="macro")
    print(f"TF-IDF C={Cc}: f1 val {f:.4f}")
    if f > mejor_f1:
        mejor_C, mejor_f1, logreg = Cc, f, lr_
pred_va = logreg.predict(X_va)
p, r, f, _ = precision_recall_fscore_support(y_todo[idx_va], pred_va, average="macro")
RES_CLF = [{"modelo": "TF-IDF + regresión logística", "fuente": "TF-IDF", "congelado": None, "mejor_epoch": None,
            "val_acc": accuracy_score(y_todo[idx_va], pred_va), "val_precision": p, "val_recall": r, "val_f1": f,
            "params_totales": logreg.coef_.size + logreg.intercept_.size,
            "params_entrenables": logreg.coef_.size + logreg.intercept_.size, "t_total": time.time() - t0}]
print(f"TF-IDF: {X_tr.shape[1]:,} características, C = {mejor_C}, {RES_CLF[0]['t_total']:.0f} s")

VARIANTES = [("Aleatorio", "Aleatorio", False), ("SGNS congelado", "SGNS", True), ("SGNS fine-tuning", "SGNS", False),
             ("gensim congelado", "gensim", True), ("gensim fine-tuning", "gensim", False),
             ("GloVe congelado", "GloVe", True), ("GloVe fine-tuning", "GloVe", False)]
HIST_CLF, MODELOS_CLF = [], {}
for nombre, fuente, congelar in VARIANTES:
    print(f"\n=== {nombre}")
    r, h, m = entrenar_clf(nombre, fuente, congelar)
    RES_CLF.append(r); HIST_CLF += h; MODELOS_CLF[nombre] = m
tabla_clf = pd.DataFrame(RES_CLF)
tabla_clf.to_csv("results/clasificacion_validacion.csv", index=False)
pd.DataFrame(HIST_CLF).to_csv("results/historial_clasificacion.csv", index=False)
tabla_clf.round(4)

In [ ]:
hc = pd.DataFrame(HIST_CLF)
fig, axs = plt.subplots(1, len(VARIANTES), figsize=(3.1 * len(VARIANTES), 2.8), sharey=True)
for ax, (nombre, _, _) in zip(axs, VARIANTES):
    h = hc[hc.modelo == nombre]
    ax.plot(h.epoch, h.perdida_train, marker="o", ms=3, color=AZUL, label="entrenamiento")
    ax.plot(h.epoch, h.perdida_val, marker="o", ms=3, color=NARANJA, label="validación")
    ax.set_title(nombre, fontsize=9, loc="left"); ax.set_xlabel("época")
axs[0].set_ylabel("pérdida"); axs[0].legend(fontsize=8, frameon=False)
fig.tight_layout(); fig.savefig("figs/curvas_clasificacion.png", bbox_inches="tight"); plt.show()

### 6.2 Evaluación final en test

In [ ]:
mejores = {"TF-IDF": "TF-IDF + regresión logística"}
for fuente in ["Aleatorio", "SGNS", "gensim", "GloVe"]:
    cand = tabla_clf[tabla_clf.fuente == fuente]
    mejores[fuente] = cand.loc[cand.val_f1.idxmax(), "modelo"]
TEST, MATRICES = {}, {}
for fuente, nombre in mejores.items():
    if fuente == "TF-IDF":
        pred = logreg.predict(X_te)
    else:
        pred = predecir(MODELOS_CLF[nombre], COD[fuente]["te"]).argmax(1).cpu().numpy()
    p, r, f, _ = precision_recall_fscore_support(y_te, pred, average="macro")
    TEST[fuente] = {"modelo": nombre, "acc": accuracy_score(y_te, pred), "precision": p, "recall": r, "f1": f}
    MATRICES[fuente] = confusion_matrix(y_te, pred).tolist()
SIMLEX_RHO = {n: spearman(e, SIMLEX) for n, e in EMB.items()}
json.dump({"test": TEST, "matrices": MATRICES, "etiquetas": ETIQUETAS, "oov": OOV, "simlex": SIMLEX_RHO},
          open("results/test_clasificacion.json", "w"), indent=2)
print("SimLex-999:", {k: round(v, 3) for k, v in SIMLEX_RHO.items()})

fig, axs = plt.subplots(1, len(MATRICES), figsize=(3.6 * len(MATRICES), 3.4))
for ax, (fuente, M) in zip(axs, MATRICES.items()):
    M = np.array(M)
    ax.imshow(M, cmap="Blues")
    for i in range(4):
        for j in range(4):
            ax.text(j, i, M[i, j], ha="center", va="center", fontsize=7, color="white" if M[i, j] > M.max() / 2 else "black")
    ax.set_xticks(range(4)); ax.set_xticklabels(ETIQUETAS, rotation=45, ha="right", fontsize=7)
    ax.set_yticks(range(4)); ax.set_yticklabels(ETIQUETAS, fontsize=7)
    ax.set_title(f"{mejores[fuente]}\nF1 {TEST[fuente]['f1']:.4f}", fontsize=8, loc="left"); ax.grid(False)
fig.tight_layout(); fig.savefig("figs/matrices_confusion.png", bbox_inches="tight"); plt.show()
pd.DataFrame(TEST).T

### 6.3 F1 de test contra fracción de los datos de entrenamiento

Cada modelo se entrena con una fracción estratificada del entrenamiento (1 % a 100 %) y siempre se selecciona la mejor época con el mismo conjunto de validación. El número de épocas crece al reducir los datos para que todos reciban una cantidad de pasos parecida (hasta 100 épocas).

In [ ]:
FRACCIONES = [0.05, 0.25, 1.0] if SMOKE else [0.01, 0.05, 0.1, 0.25, 0.5, 1.0]
CURVA = [("TF-IDF", None, None), ("Aleatorio", "Aleatorio", False), ("SGNS congelado", "SGNS", True),
         ("SGNS fine-tuning", "SGNS", False), ("GloVe congelado", "GloVe", True), ("GloVe fine-tuning", "GloVe", False)]
filas = []
y_tr_np = y_todo[idx_tr]
for frac in FRACCIONES:
    if frac < 1:
        sub, _ = train_test_split(np.arange(len(idx_tr)), train_size=frac, stratify=y_tr_np, random_state=SEED)
    else:
        sub = np.arange(len(idx_tr))
    ep = min(100, math.ceil(EPOCHS_CLF / frac))
    for nombre, fuente, congelar in CURVA:
        t0 = time.time()
        if fuente is None:
            m = LogisticRegression(C=mejor_C, max_iter=2000).fit(X_tr[sub], y_tr_np[sub])
            pred = m.predict(X_te)
        elif frac == 1.0:
            pred = predecir(MODELOS_CLF[nombre], COD[fuente]["te"]).argmax(1).cpu().numpy()
        else:
            _, _, m = entrenar_clf(nombre, fuente, congelar, torch.tensor(sub, device=DEVICE), epochs=ep, verbose=False)
            pred = predecir(m, COD[fuente]["te"]).argmax(1).cpu().numpy()
        f1 = precision_recall_fscore_support(y_te, pred, average="macro")[2]
        filas.append({"fraccion": frac, "documentos": len(sub), "modelo": nombre, "test_f1": f1, "t": time.time() - t0})
        print(f"  {100 * frac:5.1f} %  {nombre:18s} F1 test {f1:.4f}  ({time.time() - t0:.0f} s)", flush=True)
tabla_frac = pd.DataFrame(filas)
tabla_frac.to_csv("results/fracciones_ag_news.csv", index=False)
tabla_frac.pivot(index="fraccion", columns="modelo", values="test_f1").round(4)

## 7. Comparación de resultados

In [ ]:
res = {r["iteracion"]: r for r in RES}
fin = res["SGNS-final"]
ana = {}
for n, emb in EMB.items():
    ana[n] = {m: evaluar_analogias(emb, m) for m in ["add", "mul"]}
par_prom = tabla_par[tabla_par.relacion != "pares al azar"].groupby("modelo").cos_promedio.mean()
GLOVE_INFO = "6 000 M tokens; X en 85 min con 1 hilo y 14 min por iteración (300 d) con 32 núcleos Xeon E5-2658, según Pennington et al. (2014)"
filas = {
    "Tokens del corpus de entrenamiento": [f"{N_TOKENS:,}", f"{w2v.corpus_total_words:,}", "6 000 000 000"],
    "Tamaño del vocabulario": [f"{V:,}", f"{len(w2v.wv):,}", f"{len(glove.key_to_index):,}"],
    "Dimensión": [FINAL["dim"], w2v.vector_size, glove.vector_size],
    "Tiempo de entrenamiento": [f"{fin['t_total']:.0f} s en {entorno['gpu']}", f"{T_GENSIM:.0f} s en CPU con {os.cpu_count()} hilos", GLOVE_INFO],
}
for m, etq in [("add", "3CosAdd"), ("mul", "3CosMul")]:
    for k, e in [("semanticas", "semánticas"), ("sintacticas", "sintácticas"), ("total", "total")]:
        filas[f"Analogías {e}, {etq}"] = [round(ana[n][m][k], 4) for n in EMB]
filas["Spearman WordSim-353"] = [round(eval_modelos[n]["wordsim"], 4) for n in EMB]
filas["Spearman SimLex-999"] = [round(SIMLEX_RHO[n], 4) for n in EMB]
filas["Coseno promedio entre vectores diferencia"] = [round(par_prom[n], 4) for n in EMB]
filas["Tokens de AG News fuera del vocabulario (%)"] = [round(OOV[n], 2) for n in EMB]
filas["F1 macro de test en AG News"] = [round(TEST[n]["f1"], 4) for n in EMB]
comparacion = pd.DataFrame(filas, index=list(EMB)).T
comparacion.to_csv("results/comparacion.csv")
entorno["t_gensim_s"] = T_GENSIM
json.dump(entorno, open("results/entorno.json", "w"), indent=2)
comparacion

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.3))
cuerpo = [res[n] for n in ["SGNS-4", "SGNS-5", "SGNS-1"]]
for k, e, col in [("ana_total", "total", AZUL), ("ana_sem", "semánticas", VERDE), ("ana_sin", "sintácticas", NARANJA)]:
    a1.plot([r["tokens"] / 1e6 for r in cuerpo], [r[k] for r in cuerpo], marker="o", color=col, label=f"SGNS, {e}")
glove_ev = evaluar(EMB["GloVe"])
a1.axhline(glove_ev["ana_total"], color=GRIS, ls="--", lw=1.2, label=f"GloVe 6B, total ({glove_ev['ana_total']:.3f})")
a1.scatter([N_TOKENS / 1e6], [eval_modelos["gensim"]["ana_total"]], marker="s", color=MORADO, zorder=3, label="gensim, total")
a1.set_xscale("log"); a1.set_xlabel("tokens del corpus (millones, escala log)"); a1.set_ylabel("accuracy de analogías (3CosAdd)")
a1.set_title("Analogías contra tamaño del corpus", loc="left", fontsize=10); a1.legend(fontsize=8, frameon=False)

colores = {"TF-IDF": GRIS, "Aleatorio": NARANJA, "SGNS congelado": AZUL, "SGNS fine-tuning": AZUL,
           "GloVe congelado": VERDE, "GloVe fine-tuning": VERDE}
for nombre, _, _ in CURVA:
    t = tabla_frac[tabla_frac.modelo == nombre].sort_values("fraccion")
    a2.plot(100 * t.fraccion, t.test_f1, marker="o", color=colores[nombre], ls="--" if "fine" in nombre else "-", label=nombre)
a2.set_xscale("log"); a2.set_xlabel("porcentaje del entrenamiento de AG News (escala log)"); a2.set_ylabel("F1 macro en test")
a2.set_title("F1 de test contra datos de entrenamiento", loc="left", fontsize=10); a2.legend(fontsize=8, frameon=False)
fig.tight_layout(); fig.savefig("figs/comparacion.png", bbox_inches="tight"); plt.show()

## 8. Discusión

La discusión completa está en el informe PDF. Los números que la sustentan quedan en `results/`: iteraciones y curvas de SGNS, analogías individuales y por categoría, paralelismo, resultados de AG News por modelo y por fracción de datos, y la tabla comparativa.